In [5]:
import pandas as pd

file_path = "edges.csv"
df = pd.read_csv(file_path)
numero_righe = len(df)

print("Numero di righe:", numero_righe)

Numero di righe: 8100498


In [6]:
nodes_csv = "nodes.csv"
edges_csv = "edges.csv"

nodes_df = pd.read_csv(nodes_csv)
nodes_df['node_type'] = nodes_df['node_type'].str.replace('/', '_')

nodes_df.to_csv(nodes_csv, index=False)

edges_df = pd.read_csv(edges_csv)
edges_df['relation'] = edges_df['relation'].replace('off-label use', 'off_label_use')

key = edges_df[['x_index','y_index','relation']].agg(
    lambda r: frozenset([r['x_index'], r['y_index'], r['relation']]),
    axis=1
)

df2 = df.groupby(key, as_index=False).first()
df2.to_csv("edges_fixed.csv", index=False)

/tmp/ipykernel_13721/3434033994.py:19: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  df2 = df.groupby(key, as_index=False).first()


In [7]:
file_path = "edges_fixed.csv"
df = pd.read_csv(file_path)
numero_righe = len(df)

print("Numero di righe:", numero_righe)

Numero di righe: 4050064


In [ ]:
import csv
import os

def fix_csv(input_path):
    base, ext = os.path.splitext(input_path)
    output_path = base + "_fixed" + ext

    with open(input_path, newline='', encoding="utf-8", errors="replace") as fin, \
         open(output_path, "w", newline='', encoding="utf-8") as fout:

        reader = csv.reader(fin)
        writer = csv.writer(fout, quoting=csv.QUOTE_MINIMAL, escapechar='\\')

        for row in reader:
            writer.writerow(row)

fix_csv("disease_features.csv")
fix_csv("drug_features.csv")

In [ ]:
df = pd.read_csv("nodes.csv")

unique_types = df["node_type"].unique()
print(unique_types)

['gene_protein' 'drug' 'effect_phenotype' 'disease' 'biological_process'
 'molecular_function' 'cellular_component' 'exposure' 'pathway' 'anatomy']


In [ ]:
import spacy

df = pd.read_csv("nodes.csv")

allowed_types = {
    "drug",
    "biological_process",
    "molecular_function",
    "pathway",
    "effect_phenotype",
    "exposure"
}
type_map = {
    "drug": "DRUG",
    "biological_process": "BIOLOGICAL_PROCESS",
    "molecular_function": "MOLECULAR_FUNCTION",
    "pathway": "PATHWAY",
    "effect_phenotype": "EFFECT_PHENOTYPE",
    "exposure": "EXPOSURE"
}

df = df[df["node_type"].isin(allowed_types)].copy()
df["node_type"] = df["node_type"].map(type_map)


nlp = spacy.blank("en")
ruler = nlp.add_pipe("entity_ruler")

patterns = []

for _, row in df.iterrows():
    words = row['node_name'].split()
    pattern = [{"LOWER": w.lower()} for w in words]
    patterns.append({"label": row['node_type'], "pattern": pattern})

ruler.add_patterns(patterns)

nlp.to_disk("../health_ner")

print(f"Added {len(patterns)} patterns to the EntityRuler")

/home/marco/miniconda3/envs/GraphMedRag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Added 66413 patterns to the EntityRuler
